# Lesson 15: Open-Source Agent Workflows, Middleware & LangGraph

In this notebook, you will:
1. Build a `LangGraph` `StateGraph` with Conditional/Switch-Case edges and `MemorySaver` checkpointing.
2. Wrap tool calls with composable Function Middleware and expose a `/responses`-compatible handler.

In [ ]:
!pip install -q -U langgraph google-genai pydantic python-dotenv

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

class BookingState(TypedDict):
    tier: str
    status: str

def vip_node(state: BookingState):
    return {"status": "VIP Concierge Assigned"}

def std_node(state: BookingState):
    return {"status": "Standard Queue Assigned"}

def route_tier(state: BookingState):
    return "vip" if state["tier"] == "VIP" else "std"

builder = StateGraph(BookingState)
builder.add_node("vip", vip_node)
builder.add_node("std", std_node)
builder.set_conditional_entry_point(route_tier, {"vip": "vip", "std": "std"})
builder.add_edge("vip", END)
builder.add_edge("std", END)

graph = builder.compile(checkpointer=MemorySaver())
out = graph.invoke({"tier": "VIP", "status": ""}, config={"configurable": {"thread_id": "t-100"}})
print("Checkpointed LangGraph Result:", out)